<a href="https://colab.research.google.com/github/Rohitkumar6394/AI-Agent/blob/main/Voice_AI_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!apt-get install -y ffmpeg
!pip install -q openai-whisper sentence-transformers faiss-cpu transformers gtts pydub
!pip install -q pypdf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 5.4 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader

reader = PdfReader("AI_Meeting_Transcript_Sample.pdf")

transcript_text = ""

for page in reader.pages:
    transcript_text += page.extract_text() + "\n"

print(transcript_text)

AI Voice Meeting Agent - Sample Meeting
 Transcript
Manager: Good morning everyone.
Manager: The project deadline is next Friday.
Developer: We have completed the frontend module.
Tester: Testing will be finished by Wednesday.
Manager: Please make sure there are no delays.
Team Lead: We will submit the final version before Friday.
Manager: Great. Keep me updated on progress.




In [5]:
def chunk_text(text, chunk_size=400, overlap=80):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

chunks = chunk_text(transcript_text)

print("Total Chunks:", len(chunks))
print(chunks[0])

Total Chunks: 2
AI Voice Meeting Agent - Sample Meeting
 Transcript
Manager: Good morning everyone.
Manager: The project deadline is next Friday.
Developer: We have completed the frontend module.
Tester: Testing will be finished by Wednesday.
Manager: Please make sure there are no delays.
Team Lead: We will submit the final version before Friday.
Manager: Great. Keep me updated on progress.




In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype("float32"))

print("FAISS Database Ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Database Ready


In [8]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

print("LLM Ready")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deep

LLM Ready


In [10]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
import base64

def record_audio(filename="question.wav"):
    js = Javascript("""
    async function recordAudio() {
        const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        const recorder = new MediaRecorder(stream);
        let chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, 6000));

        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        const blob = new Blob(chunks, { type: 'audio/wav' });
        const reader = new FileReader();

        reader.readAsDataURL(blob);

        return await new Promise(resolve => {
            reader.onloadend = () => resolve(reader.result);
        });
    }
    """)

    display(js)
    audio_data = eval_js("recordAudio()")

    audio_bytes = base64.b64decode(audio_data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    return filename

try:
    audio_file = record_audio()
    print("Voice Recorded:", audio_file)
except Exception as e:
    print(f"Error recording audio: {e}")
    print("Please ensure microphone permissions are granted to this Colab tab in your browser.")
    print("You may need to restart the runtime after granting permissions.")
    audio_file = None

<IPython.core.display.Javascript object>

Voice Recorded: question.wav


In [11]:
import whisper

whisper_model = whisper.load_model("base")

result = whisper_model.transcribe(audio_file)

user_question = result["text"]

print("You asked:", user_question)

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 153MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


You asked: ずっとフランリしています。もっとデュマサ滝玉とねの手紙内のポジタンとゆっくり半曦 tweet


In [ ]:
def search_context(question, top_k=3):
    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        top_k
    )

    context = "\n".join([chunks[i] for i in indices[0]])

    return context

context = search_context(user_question)

print(context)

In [14]:
def search_context(question, top_k=3):
    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        top_k
    )

    context = "\n".join([chunks[i] for i in indices[0]])

    return context

context = search_context(user_question)

prompt = f"""
You are an AI Meeting Assistant.

Answer the question using only the meeting transcript context.

Context:
{context}

Question:
{user_question}

Answer:
"""

response = llm(prompt)[0]["generated_text"]

print("AI Answer:", response)

AI Answer: 
You are an AI Meeting Assistant.

Answer the question using only the meeting transcript context.

Context:
fore Friday.
Manager: Great. Keep me updated on progress.


AI Voice Meeting Agent - Sample Meeting
 Transcript
Manager: Good morning everyone.
Manager: The project deadline is next Friday.
Developer: We have completed the frontend module.
Tester: Testing will be finished by Wednesday.
Manager: Please make sure there are no delays.
Team Lead: We will submit the final version before Friday.
Manager: Great. Keep me updated on progress.


fore Friday.
Manager: Great. Keep me updated on progress.



Question:
ずっとフランリしています。もっとデュマサ滝玉とねの手紙内のポジタンとゆっくり半曦 tweet

Answer:



In [15]:
from gtts import gTTS
from IPython.display import Audio, display

tts = gTTS(response, lang="en")
tts.save("answer.mp3")

display(Audio("answer.mp3", autoplay=True))